# SmolVLA memory SFT — Kaggle 2×T4

Chọn GPU T4 ×2. Attach `memory_sft_code.zip` và prepared dataset (khuyến nghị).
Internet cần cho cài dependencies và tải checkpoint/tokenizer đã pin.
Đọc `docs/V2_MEMORY_SFT_KAGGLE.md` trong bundle trước khi chạy.
Notebook không chứa credential và không tự upload kết quả.


In [ ]:
from pathlib import Path
import os, json, subprocess, zipfile

REPO = Path("/kaggle/working/Thesis_26")
if not (REPO / "scripts/train_memory_sft.py").exists():
    bundles = list(Path("/kaggle/input").rglob("memory_sft_code.zip"))
    if len(bundles) != 1:
        raise RuntimeError("Attach đúng một memory_sft_code.zip hoặc chỉnh REPO tới source đã copy.")
    REPO.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundles[0]) as z:
        for item in z.infolist():
            if not (REPO / item.filename).resolve().is_relative_to(REPO.resolve()):
                raise ValueError("Unsafe archive path")
        z.extractall(REPO)

ENV = dict(os.environ)
ENV.update(ASSETS_DIR="/kaggle/temp/sft-assets")
def run(*args, **overrides):
    subprocess.run(["bash", str(REPO / "scripts/kaggle_memory_sft.sh"), *args],
                   cwd=REPO, env={**ENV, **overrides}, check=True)
print(REPO)


## 1. Môi trường training riêng và assets

In [ ]:
run("install")
run("assets")


## 2. Chọn dữ liệu

Khuyến nghị attach prepared dataset để không chiếm /kaggle/working.
Nếu chưa có, đặt `PREPARE_ON_KAGGLE=True` và kiểm tra dung lượng: raw full suite 13.73 GB,
chưa gồm prepared shards, môi trường và checkpoints. `TASK_NAMES` có thể giới hạn pilot.


In [ ]:
PREPARE_ON_KAGGLE = False
DATA_DIR = Path("/kaggle/input/libero10-memory")  # chỉnh theo dataset bạn attach
TASK_NAMES = []  # ví dụ: ["STUDY_SCENE1_pick_up_the_book_and_place_it_in_the_back_compartment_of_the_caddy"]

if PREPARE_ON_KAGGLE:
    DATA_DIR = Path("/kaggle/working/libero10-memory")
    extra = ["--task-names", *TASK_NAMES] if TASK_NAMES else []
    run("prepare", "--source-lock", "resources/manifests/libero_10_sft_sources.json",
        *extra, DATA_DIR=str(DATA_DIR))
if not (DATA_DIR / "manifest.json").is_file():
    raise FileNotFoundError("Chỉnh DATA_DIR tới thư mục chứa manifest.json và shards/.")
ENV["DATA_DIR"] = str(DATA_DIR)
manifest = json.loads((DATA_DIR / "manifest.json").read_text())
print("Suite:", manifest["suite"], "Source:", manifest["source_revision"])
for row in manifest["sources"]:
    print(row["path"], row["frame_memory_counts_by_split"])


## 3. Smoke test trên đúng hai T4 (2 optimizer updates)

In [ ]:
run("train", "--steps", "2", "--warmup-steps", "0",
    "--validation-max-batches", "2", "--save-every", "1",
    OUTPUT_DIR="/kaggle/working/sft-smoke")


## 4. Training

Preset: frozen VLM, expert LoRA r16, FP16, batch2/GPU, accumulation8, effective batch32.
Nếu OOM ở smoke, dùng `--batch-size 1 --accumulation 16` cho cả hai run mới.
Đối chứng: đổi `CONDITION` thành `no_memory` và dùng output khác.


In [ ]:
RUN_TRAINING = False  # đổi True khi smoke đã qua
CONDITION = "memory"
OUTPUT_DIR = f"/kaggle/working/sft-{CONDITION}"
if RUN_TRAINING:
    run("train", "--condition", CONDITION, "--adaptation", "expert_lora", "--steps", "3000",
        OUTPUT_DIR=OUTPUT_DIR)


## Resume và dùng checkpoint

Resume bằng cùng command/arguments và thêm `--resume /.../checkpoint-001000`.
Không đổi steps/batch/schedule của run gốc. Không dùng smoke checkpoint làm resume cho long run.

Kết quả cuối: `OUTPUT_DIR/pretrained_model/`, LoRA đã merge và normalizers giữ nguyên.
Tải cả thư mục về máy đánh giá rồi chạy:

```bash
python scripts/memory_sft_assets.py relocate /absolute/path/to/pretrained_model
```

Validation flow loss không phải success rate. Đánh giá lại với protocol và execution horizon10 đã khoá.
